# 02 — Fetch isochrones

Asks Mapbox, for each open branch, "what area can you reach in N minutes?" and
stores the answer as a polygon.

| | |
|---|---|
| **Reads** | `clean/branches.csv`, Feast online store |
| **Writes** | `features/isochrones.parquet`, Feast online store |
| **Needs** | `cpl-s3`, `mapbox-token` |

Same behaviour as the `cpl-fetch-isochrones-*` pipelines, including the Feast
cache. Two things are handled differently here, both explained where they
happen: the Feast config is written by the notebook instead of mounted, and
the client version is pinned.

### Running this as an Elyra node

| Property | Value |
|---|---|
| Runtime image | Datascience with Python 3.12 |
| Kubernetes secret | `cpl-s3` → `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, `AWS_S3_BUCKET` |
| Kubernetes secret | `mapbox-token` → `MAPBOX_TOKEN` |
| Environment variables | `MODE`, `TRAVEL_TIME` |

**This is the notebook to put on the canvas three times.** `MODE` and
`TRAVEL_TIME` come from the environment, so one file covers all three travel
modes — give each copy different values and chain them in series:

| Node | `MODE` | `TRAVEL_TIME` |
|---|---|---|
| driving | `driving` | `5` |
| cycling | `cycling` | `10` |
| walking | `walking` | `10` |

Chain them rather than running them side by side. Mapbox allows 300 requests a
minute and three parallel nodes will trip it — which is exactly why the KFP
version runs its three pipelines sequentially too.

In [ ]:
%pip install -q boto3 requests shapely pandas pyarrow "feast[grpcio]==0.65.0"

**Why the Feast version is pinned.** A plain `pip install feast` resolves to an
older client in this image, and an older client talking to this feature server
fails every call with a `422`. The failure is caught and logged rather than
raised, so the symptom is not an error — it is a cache that silently never
hits and a pipeline that refetches all 80 branches from Mapbox on every run.

The pin has to match the server. To check what is deployed:

```
oc get featurestore -n cpl-coverage -o jsonpath='{.items[0].status.feastVersion}'
```

In [ ]:
import io
import os
import logging

import boto3
import pandas as pd

logging.basicConfig(level=logging.INFO, force=True)
log = logging.getLogger("fetch_isochrones")

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    verify=False,
)
BUCKET = os.environ["AWS_S3_BUCKET"]
log.info("Bucket: %s", BUCKET)

## Parameters

Change these inline when running interactively. As an Elyra node the
environment wins, so each copy on the canvas can target a different mode.

In [ ]:
MODE = os.environ.get("MODE", "driving")
TRAVEL_TIME = int(os.environ.get("TRAVEL_TIME", "5"))

# Skip any branch fetched more recently than this. Traffic does not move much
# inside an hour, and the Mapbox free tier is 100k requests/month.
FRESHNESS_HOURS = float(os.environ.get("FRESHNESS_HOURS", "1"))

RATE_LIMIT_DELAY = 0.25  # Mapbox allows 300 req/min
MAX_RETRIES = 3

MAPBOX_TOKEN = os.environ["MAPBOX_TOKEN"]
CACHE_KEY = "features/isochrones.parquet"
FEAST_STORE = "cpl-features"  # the FeatureStore CR name

log.info("Mode: %s, travel time: %d min", MODE, TRAVEL_TIME)

## Connect to Feast

The KFP version gets its `feature_store.yaml` from the `feast-client-config`
ConfigMap, mounted at `/feast-config`. An Elyra node cannot mount a ConfigMap —
node properties cover environment variables, secrets, PVCs, annotations, labels
and tolerations, and that is the whole list.

It does not need to. The file is twelve lines and the notebook can write it
out itself. The TLS certificate it references is the OpenShift service CA,
which is already projected into *every* pod at
`/var/run/secrets/kubernetes.io/serviceaccount/service-ca.crt` — byte for byte
the same bundle the ConfigMap carries. The namespace comes from the same place,
so nothing here is hardcoded to one cluster.

In [ ]:
import tempfile
from pathlib import Path

SA = Path("/var/run/secrets/kubernetes.io/serviceaccount")
NAMESPACE = SA.joinpath("namespace").read_text().strip()
CA_CERT = str(SA / "service-ca.crt")

feast_repo = Path(tempfile.mkdtemp(prefix="feast-"))
(feast_repo / "feature_store.yaml").write_text(f"""\
project: cpl_coverage
provider: local
online_store:
    path: https://feast-{FEAST_STORE}-online.{NAMESPACE}.svc.cluster.local:443
    type: remote
    cert: {CA_CERT}
registry:
    registry_type: remote
    path: feast-{FEAST_STORE}-registry.{NAMESPACE}.svc.cluster.local:443
    cert: {CA_CERT}
auth:
    type: no_auth
entity_key_serialization_version: 3
""")

from feast import FeatureStore
from feast.data_source import PushMode

store = FeatureStore(repo_path=str(feast_repo))
log.info("Feast connected to project %s in namespace %s", store.project, NAMESPACE)
log.info("Feature views: %s", [fv.name for fv in store.list_feature_views()])

## Load branches

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key="clean/branches.csv")
branches = pd.read_csv(io.BytesIO(obj["Body"].read()))
branches = branches[branches["PERMANENTLY_CLOSED"] == 0].reset_index(drop=True)
log.info("Loaded %d active branches", len(branches))

branches[["BRANCH", "LATITUDE", "LONGITUDE"]].head()

## Ask Feast which branches are still fresh

The KFP component queries Feast once per branch, inside the fetch loop. This
does the same lookup for every branch in one call before the loop starts —
same decision, 1 round trip instead of 80.

If Feast is unreachable the lookup is logged as a **warning** and every branch
is treated as stale. That is deliberate: a silent failure here looks exactly
like a working cache that happens to be cold, and you would never notice you
were paying Mapbox for data you already had.

In [ ]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)
cutoff_ts = int((now - timedelta(hours=FRESHNESS_HOURS)).timestamp())

fresh_branches = set()
try:
    names = branches["BRANCH"].tolist()
    feast_result = store.get_online_features(
        features=["isochrone_features:fetch_ts"],
        entity_rows=[{"isochrone_key": f"{n}|{MODE}|{TRAVEL_TIME}"} for n in names],
    ).to_dict()

    for name, ts in zip(names, feast_result["fetch_ts"]):
        if ts is not None and ts >= cutoff_ts:
            fresh_branches.add(name)

    log.info("Feast reports %d/%d branches still fresh for %s/%dmin",
             len(fresh_branches), len(names), MODE, TRAVEL_TIME)
except Exception as e:
    log.warning("Feast lookup failed, treating every branch as stale: %s", e)

## Fetch

One request per stale branch. Three tries each, because a single timeout
should not cost the whole run.

In [ ]:
import time

import requests
from shapely.geometry import shape

base_url = f"https://api.mapbox.com/isochrone/v1/mapbox/{MODE}/"

new_rows = []
stats = {"cached": 0, "fetched": 0, "failed": 0}

for _, branch in branches.iterrows():
    branch_name = branch["BRANCH"]
    lat, lon = branch["LATITUDE"], branch["LONGITUDE"]

    if branch_name in fresh_branches:
        stats["cached"] += 1
        continue

    success = False
    for attempt in range(MAX_RETRIES):
        try:
            url = (
                f"{base_url}{lon},{lat}"
                f"?contours_minutes={TRAVEL_TIME}"
                f"&polygons=true&access_token={MAPBOX_TOKEN}"
            )
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            geometry = resp.json()["features"][0]["geometry"]

            new_rows.append({
                "isochrone_key": f"{branch_name}|{MODE}|{TRAVEL_TIME}",
                "branch": branch_name,
                "mode": MODE,
                "travel_time": TRAVEL_TIME,
                "latitude": lat,
                "longitude": lon,
                "geometry_wkt": shape(geometry).wkt,
                "fetch_ts": int(now.timestamp()),
                "event_timestamp": now,
            })
            stats["fetched"] += 1
            success = True
            break
        except Exception as e:
            log.warning("Attempt %d/%d failed for %s (%s, %dmin): %s",
                        attempt + 1, MAX_RETRIES, branch_name, MODE, TRAVEL_TIME, e)
            if attempt < MAX_RETRIES - 1:
                time.sleep(1)

    if not success:
        stats["failed"] += 1
        log.error("All %d attempts failed for %s (%s, %dmin)",
                  MAX_RETRIES, branch_name, MODE, TRAVEL_TIME)

    time.sleep(RATE_LIMIT_DELAY)

log.info("Cached: %(cached)d, Fetched: %(fetched)d, Failed: %(failed)d", stats)

## Push to Feast

This is what makes the *next* run cheap. Without it the lookup above always
comes back empty and every branch is refetched forever.

In [ ]:
if new_rows:
    push_df = pd.DataFrame(new_rows)
    push_df["event_timestamp"] = pd.to_datetime(push_df["event_timestamp"], utc=True)
    try:
        store.push("isochrone_push", push_df, to=PushMode.ONLINE)
        log.info("Pushed %d isochrones to Feast", len(push_df))
    except Exception as e:
        log.warning("Feast push failed, cache will stay cold: %s", e)
else:
    log.info("Nothing new to push")

## Append to the S3 cache

Feast holds only the newest value per key. The parquet file is append-only and
keeps every fetch, which is what lets step 03 pick a point in time and what
would let you see a branch's reachable area change as roadworks come and go.

In [ ]:
try:
    obj = s3.get_object(Bucket=BUCKET, Key=CACHE_KEY)
    existing = pd.read_parquet(io.BytesIO(obj["Body"].read()))
except Exception as e:
    log.info("No parquet cache yet (%s)", type(e).__name__)
    existing = pd.DataFrame()

if new_rows:
    new_df = pd.DataFrame(new_rows)
    new_df["event_timestamp"] = pd.to_datetime(new_df["event_timestamp"], utc=True)
    result = pd.concat([existing, new_df], ignore_index=True) if not existing.empty else new_df
else:
    result = existing

if not result.empty:
    buf = io.BytesIO()
    result.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=CACHE_KEY, Body=buf.getvalue())

log.info("Done (%s, %dmin). Cache now holds %d rows.", MODE, TRAVEL_TIME, len(result))
result.groupby(["mode", "travel_time"]).size()